In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
# SOURCE AND TARGET PATHS

constructors_source_path = f"{s3_root_path}/Silver/Aaisha/constructors"
ref_nationality_region_path = f"{s3_root_path}/Gold/Aaisha/ref_nationality_region"

target_path = f"{s3_root_path}/Gold/Aaisha/dim_constructors"

print("Constructors Source :", constructors_source_path)
print("Reference Source    :", ref_nationality_region_path)
print("Target Path         :", target_path)

In [0]:
constructors_df = (
    spark.read
        .format("delta")
        .load(constructors_source_path)
)

ref_nationality_region_df = (
    spark.read
        .format("delta")
        .load(ref_nationality_region_path)
)

In [0]:
dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region_df,
            constructors_df.nationality == ref_nationality_region_df.nationality,
            "left"
        )
        .select(
            constructors_df.constructorId.alias("constructor_id"),
            constructors_df.name.alias("constructor_name"),
            constructors_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

display(dim_constructors_df)

In [0]:
(
    dim_constructors_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
gold_constructors_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_constructors_df)